# Assignment 3 – Pandas 2
### Data Cleaning | Missing Values | GroupBy & Aggregations | Reshaping | Rolling & Cumulative Operations | Joins and Concatenation

Complete answers and runnable Google Colab code for all 10 questions.


## Question 1 – Data Cleaning and Missing Values

### Answer
**Data cleaning** is the process of detecting, correcting, or removing inaccurate, incomplete, inconsistent, or irrelevant data before analysis. It is performed because poor-quality data can produce misleading summaries, visualizations, and machine-learning results.

In Pandas:
- `isna().sum()` counts the number of missing values in each column.
- `isna().mean()` gives the fraction of values that are missing in each column. Multiplying it by 100 gives the percentage.

A column may be dropped instead of imputed when a very large proportion of its values are missing, when the column has little analytical value, or when a reliable imputation method is not available. Dropping should be considered carefully because it can remove useful information.


In [ ]:
# Q1 demonstration
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "age": [20, np.nan, 25, 30, np.nan],
    "salary": [30000, 35000, np.nan, 40000, 45000],
    "city": ["Hyderabad", "Delhi", np.nan, "Chennai", "Mumbai"]
})

print("Number of missing values:")
print(df.isna().sum())

print("\nFraction of missing values:")
print(df.isna().mean())

print("\nPercentage of missing values:")
print(df.isna().mean() * 100)


## Question 2 – Median and Mode Imputation

### Answer
**Median imputation** replaces missing numerical values with the median of the available values. The median is preferred for numerical variables such as `age` because it is less affected by extreme values and outliers than the mean.

**Mode imputation** replaces missing categorical values with the most frequently occurring category. It is appropriate for categorical columns such as `embarked` because categories are not numerical quantities for which a mean or median would make sense.

Example:
```python
df['age'] = df['age'].fillna(df['age'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])
```


In [ ]:
# Q2
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "age": [20, 25, np.nan, 30, 100],
    "embarked": ["S", "C", "S", np.nan, "S"]
})

df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])

print(df)


## Question 3 – `map()` vs `apply()`

### Answer
`map()` is commonly used with a Pandas Series to transform each value individually. A dictionary can be supplied to map categories to new values.

`apply()` is more general. It can apply a function to each element of a Series or along an axis of a DataFrame.

Example:
```python
df['grade_code'] = df['grade'].map({'A': 1, 'B': 2, 'C': 3})
df['age_plus_one'] = df['age'].apply(lambda x: x + 1)
```

When a value is **missing from the dictionary passed to `map()`**, its mapped result becomes `NaN` (unless the mapping is implemented in a way that supplies another fallback).


In [ ]:
# Q3
import pandas as pd

df = pd.DataFrame({
    "grade": ["A", "B", "C", "D"],
    "age": [20, 25, 30, 35]
})

# map() with a dictionary
grade_map = {"A": 1, "B": 2, "C": 3}
df["grade_code"] = df["grade"].map(grade_map)

# apply() with a function
df["age_plus_one"] = df["age"].apply(lambda x: x + 1)

print(df)
print("\nNotice that D is not in the dictionary, so its grade_code is NaN.")


## Question 4 – Binning with `cut()` and `qcut()`

### Answer
**Binning** converts continuous numerical values into discrete intervals or categories.

`pd.cut()` divides the numerical range into specified intervals. The bins are based on value ranges, so the number of observations in each bin may differ.

Example:
```python
pd.cut(df['age'], bins=[0, 18, 35, 60, 100])
```

Real-world use: age groups such as child, young adult, adult, and senior.

`pd.qcut()` divides values into approximately equal-sized groups based on quantiles. The interval widths can differ because the goal is to distribute observations across the groups.

Example:
```python
pd.qcut(df['income'], q=4)
```

Real-world use: dividing customers into income or spending quartiles for segmentation.


In [ ]:
# Q4
import pandas as pd

df = pd.DataFrame({
    "age": [5, 12, 20, 28, 40, 55, 70, 80],
    "income": [20, 25, 30, 35, 50, 70, 90, 120]
})

df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 18, 35, 60, 100],
    labels=["Child", "Young Adult", "Adult", "Senior"]
)

df["income_quartile"] = pd.qcut(
    df["income"],
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

print(df)


## Question 5 – `agg()`, `transform()` and `filter()`

### Answer
In a `groupby` operation:

- **`agg()`** produces summary statistics for each group, such as sum, mean, minimum, or maximum. The result usually has one row per group.
- **`transform()`** calculates a group-level value but returns it aligned to the original rows, making it useful for adding group statistics as new columns.
- **`filter()`** keeps or removes entire groups based on a condition.

Examples:
```python
grouped.agg(...)
grouped.transform(...)
grouped.filter(...)
```


In [ ]:
# Q5
import pandas as pd

df = pd.DataFrame({
    "region": ["North", "North", "South", "South", "East"],
    "sales": [100, 150, 200, 250, 120]
})

grouped = df.groupby("region")

# agg(): one summary row per group
summary = grouped["sales"].agg(["sum", "mean"])
print("agg():")
print(summary)

# transform(): group mean aligned with original rows
df["region_mean"] = grouped["sales"].transform("mean")
print("\ntransform():")
print(df)

# filter(): keep groups whose total sales exceeds 250
filtered = grouped.filter(lambda g: g["sales"].sum() > 250)
print("\nfilter():")
print(filtered)


## Question 6 – GroupBy and Named Aggregation

### Dataset
The question uses the specified region/category/sales/discount/quantity dataset.

### Answer
The program below:
1. Groups by `region` and `category` and calculates total sales.
2. Counts records in each group.
3. Creates a region-wise named-aggregation summary with `total_sales`, `avg_discount`, and `max_qty`.
4. Calculates mean, minimum, and maximum sales for each region-category pair.


In [ ]:
# Q6
import pandas as pd

df = pd.DataFrame({
    'region': ['North','North','South','South','East','East','East'],
    'category': ['A','B','A','A','B','A','B'],
    'sales': [100, 150, 200, 250, 120, 90, 300],
    'discount': [5, 10, 0, 2, 3, 1, 4],
    'quantity': [10, 12, 15, 20, 8, 9, 14]
})

# 1. Total sales of every region-category group
total_sales = df.groupby(["region", "category"])["sales"].sum().reset_index()
print("1. Total sales by region and category:")
print(total_sales)

# 2. Number of records in every group
record_count = df.groupby(["region", "category"]).size().reset_index(name="record_count")
print("\n2. Number of records:")
print(record_count)

# 3. Region-wise named aggregation
region_summary = df.groupby("region").agg(
    total_sales=("sales", "sum"),
    avg_discount=("discount", "mean"),
    max_qty=("quantity", "max")
).reset_index()

print("\n3. Region-wise summary:")
print(region_summary)

# 4. Mean, min and max sales for every region-category pair
sales_stats = df.groupby(["region", "category"])["sales"].agg(
    mean_sales="mean",
    min_sales="min",
    max_sales="max"
).reset_index()

print("\n4. Mean, min and max sales:")
print(sales_stats)


## Question 7 – `transform()`, Contribution, Ranking and Group Filtering

### Answer
`transform()` is useful here because the calculated values must remain aligned with the original rows.

- `region_avg_sales`: average sales for the row's region.
- `pct_contribution`: row sales divided by total sales of its region, multiplied by 100.
- `rank_region_sales`: descending sales rank within each region.
- The final filter keeps only regions containing more than two records.


In [ ]:
# Q7
import pandas as pd

df = pd.DataFrame({
    'region': ['North','North','South','South','East','East','East'],
    'category': ['A','B','A','A','B','A','B'],
    'sales': [100, 150, 200, 250, 120, 90, 300],
    'discount': [5, 10, 0, 2, 3, 1, 4],
    'quantity': [10, 12, 15, 20, 8, 9, 14]
})

# 1. Average sales of each region
df["region_avg_sales"] = df.groupby("region")["sales"].transform("mean")

# 2. Percentage contribution to regional total
region_total = df.groupby("region")["sales"].transform("sum")
df["pct_contribution"] = (df["sales"] / region_total) * 100

# 3. Rank sales within each region, descending
df["rank_region_sales"] = df.groupby("region")["sales"].rank(
    ascending=False,
    method="dense"
)

print("Data with transformed columns:")
print(df)

# 4. Keep only regions with more than two records
result = df.groupby("region").filter(lambda g: len(g) > 2)

print("\nRegions containing more than two records:")
print(result)


## Question 8 – Pivot and Pivot Table

### Dataset
The question uses the specified region/month/sales/discount dataset.

### Answer
`pivot()` reshapes data when each combination of index and columns identifies a unique value. It does not aggregate duplicate combinations.

`pivot_table()` is more flexible because it can aggregate duplicate combinations using an aggregation function such as `sum` or `mean`.

Therefore, `pivot()` fails when there are duplicate entries for the same `index` + `columns` combination, while `pivot_table()` can handle them by aggregating the duplicate values.


In [ ]:
# Q8
import pandas as pd

df = pd.DataFrame({
    'region': ['North','North','South','South','East','East'],
    'month': ['Jan','Feb','Jan','Feb','Jan','Feb'],
    'sales': [100, 120, 200, 220, 150, 180],
    'discount': [5, 7, 3, 4, 6, 8]
})

# 1. Wide format using pivot()
wide = df.pivot(index="region", columns="month", values="sales")
print("1. Wide format using pivot():")
print(wide)

# 2. Pivot table of total sales with Total row and column
sales_pivot = pd.pivot_table(
    df,
    index="region",
    columns="month",
    values="sales",
    aggfunc="sum",
    margins=True,
    margins_name="Total"
)
print("\n2. Total sales pivot table:")
print(sales_pivot)

# 3. Average discount, filling missing combinations with 0
discount_pivot = pd.pivot_table(
    df,
    index="region",
    columns="month",
    values="discount",
    aggfunc="mean",
    fill_value=0
)
print("\n3. Average discount pivot table:")
print(discount_pivot)

# 4. Demonstrate why pivot() fails with duplicate combinations
duplicate_df = pd.DataFrame({
    "region": ["North", "North"],
    "month": ["Jan", "Jan"],
    "sales": [100, 150]
})

try:
    print(duplicate_df.pivot(index="region", columns="month", values="sales"))
except ValueError as e:
    print("\n4. pivot() with duplicate combinations raises:")
    print(e)

print("\npivot_table() can aggregate duplicates:")
print(pd.pivot_table(
    duplicate_df,
    index="region",
    columns="month",
    values="sales",
    aggfunc="sum"
))


## Question 9 – Rolling, Expanding and Cumulative Operations

### Answer
- **Rolling operation:** calculates a statistic over a moving window of a fixed size. For example, a 3-period rolling mean uses the current value and the previous two values.
- **Expanding operation:** calculates a statistic using all values from the beginning up to the current position.
- **Cumulative operation:** calculates a running result, such as cumulative sum or cumulative maximum.

With a 3-period rolling mean, the first two values are `NaN` by default because there are not yet three observations available to form a complete window.

With `min_periods=1`, Pandas allows the rolling calculation to use whatever number of observations is available, so the first value uses one observation and the second uses two.


In [ ]:
# Q9
import pandas as pd

df = pd.DataFrame({
    'region': ['North','North','South','South','East','East'],
    'month': ['Jan','Feb','Jan','Feb','Jan','Feb'],
    'sales': [100, 120, 200, 220, 150, 180],
    'discount': [5, 7, 3, 4, 6, 8]
})

df["rolling_mean_3"] = df["sales"].rolling(window=3).mean()
df["expanding_avg"] = df["sales"].expanding().mean()
df["cum_sum"] = df["sales"].cumsum()
df["cum_max"] = df["sales"].cummax()

print("Default rolling window:")
print(df)

# Demonstrate min_periods=1
df["rolling_mean_3_min1"] = df["sales"].rolling(
    window=3,
    min_periods=1
).mean()

print("\nRolling mean with min_periods=1:")
print(df[["sales", "rolling_mean_3", "rolling_mean_3_min1"]])


## Question 10 – Complete Sales/Representatives Analysis

### Answer
The program performs all requested operations in one workflow:
1. Displays missing values and fills missing `sales` with the median.
2. Performs a left merge with the representative table using `rep_id`.
3. Calculates `revenue_per_unit`.
4. Creates a region-wise summary with total sales, average sales, and order count.
5. Creates a pivot table of total sales by region and category.
6. Groups by representative and identifies the representative with the highest total sales.


In [ ]:
# Q10
import pandas as pd
import numpy as np

sales = pd.DataFrame({
    'order_id': [1, 2, 3, 4, 5, 6],
    'region': ['North', 'South', 'East', 'North', 'South', 'East'],
    'category': ['A', 'B', 'A', 'B', 'A', 'B'],
    'sales': [100, 250, np.nan, 160, 200, 300],
    'quantity': [10, 20, 12, 10, 15, 14],
    'rep_id': [101, 102, 103, 101, 102, 103]
})

reps = pd.DataFrame({
    'rep_id': [101, 102, 103],
    'rep_name': ['Aditi', 'Rahul', 'Sneha'],
    'experience': [5, 2, 7]
})

# 1. Display missing values and fill missing sales with median
print("Missing values before filling:")
print(sales.isna().sum())

sales["sales"] = sales["sales"].fillna(sales["sales"].median())

print("\nSales after median imputation:")
print(sales)

# 2. Left join with representatives
merged = pd.merge(
    sales,
    reps,
    on="rep_id",
    how="left"
)

# 3. Revenue per unit
merged["revenue_per_unit"] = merged["sales"] / merged["quantity"]

print("\nMerged data:")
print(merged)

# 4. Region-wise summary
region_summary = merged.groupby("region").agg(
    total_sales=("sales", "sum"),
    avg_sales=("sales", "mean"),
    orders=("order_id", "count")
).reset_index()

print("\nRegion-wise summary:")
print(region_summary)

# 5. Pivot table of total sales
sales_pivot = pd.pivot_table(
    merged,
    index="region",
    columns="category",
    values="sales",
    aggfunc="sum",
    fill_value=0
)

print("\nPivot table of total sales:")
print(sales_pivot)

# 6. Representative with highest total sales
rep_sales = merged.groupby("rep_name")["sales"].sum()
top_rep = rep_sales.idxmax()

print("\nTotal sales by representative:")
print(rep_sales)

print("\nRepresentative with highest total sales:", top_rep)


# End of Assignment

Run the code cells in Google Colab to see the outputs.